# 04. Simulated Annealing 실행

`dwave-neal`을 사용하여 QUBO를 직접 해결한다. QA와의 비교가 공정하도록 **동일한 QUBO**를 사용하며, 파라미터는 설정 파일에 고정하고 결과를 보고 튜닝하지 않는다.

best sample 하나만 보지 않고, `num_reads` 안에서 feasible sample 비율과 feasible 해 중 최선의 목적값도 함께 기록하여 solver의 안정성을 평가한다.

**주의**: dimod `SampleSet.samples()`는 정렬된 뷰를 반환하므로 `record.energy`와 인덱스가 어긋날 수 있다. 본 프로젝트는 항상 `record.sample`과 `record.energy`를 같은 인덱스로 함께 읽는다.

In [1]:
# 프로젝트 루트를 import 경로에 추가한다.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"
SOLUTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("설정 로드 완료:", len(config["instances"]), "개 instance")


설정 로드 완료: 4 개 instance


In [2]:
from src import sa_solver
from src.data_generator import CFLPInstance
from src.persistence import save_samples, save_table
from src.qubo_builder import build_qubo

margin = float(config["penalty"]["margin"])
precision = int(config["encoding"]["precision"])
feas_tolerance = float(config["feasibility"]["tolerance"])

instances = [
    CFLPInstance.load(DATA_DIR / f"{spec['name']}.json")
    for spec in config["instances"]
]

# linking constraint 포함/제외 두 변형을 모두 실행한다.
# x_ij <= y_j 는 capacity constraint에 의해 함의되는 redundant 제약이므로
# feasible region은 동일하다. 따라서 관측되는 차이는 전부 QUBO 표현의
# 비용이며, 이것이 이 비교의 목적이다.
LINKING_VARIANTS = (False, True)

records = []
best_samples = {}
for instance in instances:
    for formulation in ("SS", "MS"):
      for include_linking in LINKING_VARIANTS:
        model = build_qubo(
            instance, formulation, margin, precision,
            include_linking=include_linking,
        )
        outcome = sa_solver.solve(model, instance, config["sa"], feas_tolerance)
        record = {
            "instance": instance.name,
            "size": instance.num_customers,
            "formulation": formulation,
            "linking": include_linking,
            "qubo_variables": model.num_variables,
            "qubo_quadratic_terms": model.num_quadratic_terms,
        }
        record.update(outcome.to_record())
        records.append(record)
        tag = "L" if include_linking else "N"
        key = f"{instance.name}__{formulation}__{tag}"
        if outcome.best_sample is not None:
            best_samples[key + "__best"] = outcome.best_sample
        if outcome.best_feasible_sample is not None:
            best_samples[key + "__best_feasible"] = outcome.best_feasible_sample
        print(
            f"{instance.name} {formulation} linking={str(include_linking):5s}: "
            f"vars={model.num_variables:5d}, "
            f"obj={outcome.best_objective:.2f}, "
            f"feasible_fraction={outcome.feasible_fraction:.3f}, "
            f"time={outcome.runtime:.1f}s"
        )

sa_results = pd.DataFrame(records)
save_table(sa_results, RAW_DIR, "sa_results.csv")
save_samples(best_samples, RAW_DIR, "sa_best_samples.npz")
sa_results

4x4 SS linking=False: vars=   44, obj=2593.09, feasible_fraction=0.009, time=0.4s
4x4 SS linking=True : vars=   60, obj=2416.84, feasible_fraction=0.008, time=0.6s
4x4 MS linking=False: vars=  120, obj=3123.42, feasible_fraction=0.897, time=2.2s
4x4 MS linking=True : vars=  212, obj=3578.58, feasible_fraction=0.630, time=3.5s
6x6 SS linking=False: vars=   79, obj=5354.13, feasible_fraction=0.002, time=0.7s
6x6 SS linking=True : vars=  115, obj=5487.61, feasible_fraction=0.003, time=1.0s
6x6 MS linking=False: vars=  259, obj=5193.73, feasible_fraction=0.977, time=6.5s
6x6 MS linking=True : vars=  475, obj=6031.71, feasible_fraction=0.653, time=9.5s
8x8 SS linking=False: vars=  117, obj=5725.81, feasible_fraction=0.002, time=1.2s
8x8 SS linking=True : vars=  181, obj=6552.75, feasible_fraction=0.003, time=1.7s
8x8 MS linking=False: vars=  413, obj=6683.23, feasible_fraction=0.909, time=11.8s
8x8 MS linking=True : vars=  773, obj=6897.26, feasible_fraction=0.597, time=17.6s
15x15 SS linki

,instance,size,formulation,linking,qubo_variables,qubo_quadratic_terms,solver,status,runtime,num_reads,best_energy,best_objective,best_is_feasible,best_total_violation,best_max_violation,feasible_fraction,best_feasible_objective,best_feasible_energy,num_samples,energy_mismatch,reported_best_energy,argmin_agreement,unique_samples,feasible_reads,sa_num_sweeps,sa_seed
0,4x4,4,SS,False,44,246,SA,OK,0.419581,1000,2593.087900,2593.0879,True,0.0,0.0,0.009,2593.0879,2593.087900,1000,7.996406e-15,2593.087900,True,1000,9,1000,12345
1,4x4,4,SS,True,60,278,SA,OK,0.568637,1000,2416.836400,2416.8364,True,0.0,0.0,0.008,2416.8364,2416.836400,1000,1.002223e-14,2416.836400,True,1000,8,1000,12345
2,4x4,4,MS,False,120,2540,SA,OK,2.206470,1000,3123.415500,3123.4155,True,0.0,0.0,0.897,3123.4155,3123.415500,1000,4.047188e-14,3123.415501,True,1000,897,1000,12345
3,4x4,4,MS,True,212,3384,SA,OK,3.500570,1000,3578.583300,3578.5833,True,0.0,0.0,0.630,3578.5833,3578.583300,1000,7.176949e-14,3578.583298,True,1000,630,1000,12345
4,6x6,6,SS,False,79,571,SA,OK,0.734470,1000,18324.893920,5354.1282,False,1.0,1.0,0.002,5988.4188,18959.184520,1000,5.836658e-14,18324.893923,True,1000,2,1000,12345
5,6x6,6,SS,True,115,643,SA,OK,1.045273,1000,18458.380420,5487.6147,False,1.0,1.0,0.003,5711.6042,874752.907440,1000,6.631337e-14,18458.380427,True,1000,3,1000,12345
6,6x6,6,MS,False,259,8701,SA,OK,6.466007,1000,5193.725600,5193.7256,True,0.0,0.0,0.977,5193.7256,5193.725600,1000,6.304288e-13,5193.725621,True,1000,977,1000,12345
7,6x6,6,MS,True,475,10753,SA,OK,9.536324,1000,6031.706302,6031.7063,True,0.0,0.0,0.653,5930.6206,31872.152040,1000,5.038570e-13,6031.706350,True,1000,653,1000,12345
8,8x8,8,SS,False,117,1030,SA,OK,1.185111,1000,5725.805300,5725.8053,True,0.0,0.0,0.002,5271.9611,95394.894000,1000,5.095009e-14,5725.805321,True,1000,2,1000,12345
9,8x8,8,SS,True,181,1158,SA,OK,1.715565,1000,24577.338980,6552.7524,False,1.0,1.0,0.003,5962.6765,78061.022820,1000,2.353065e-14,24577.338973,True,1000,3,1000,12345


## sample-energy 짝맞춤 교차검증

우리가 직접 계산한 QUBO energy와 sampler가 보고한 energy가 일치하는지 확인한다. best 하나가 아니라 **모든 sample**에 대해 비교하며, energy 최소값의 위치(argmin)가 일치하는지도 확인한다.

penalty 항이 `10^9` 규모에서 `10^4`까지 상쇄되므로, 상대오차의 분모로는 energy가 아니라 QUBO 계수의 최대 절댓값을 쓴다. 즉 "실제 합산 정밀도" 기준으로 본다.

In [3]:
mismatch = sa_results[
    ["instance", "formulation", "energy_mismatch", "argmin_agreement"]
].copy()
print("최대 상대 mismatch:", float(mismatch["energy_mismatch"].max()))
print("argmin 일치:", bool(mismatch["argmin_agreement"].all()))
assert float(mismatch["energy_mismatch"].max()) < 1e-6
assert bool(mismatch["argmin_agreement"].all())
mismatch

최대 상대 mismatch: 2.7476089534171312e-11
argmin 일치: True


,instance,formulation,energy_mismatch,argmin_agreement
0,4x4,SS,7.996406e-15,True
1,4x4,SS,1.002223e-14,True
2,4x4,MS,4.047188e-14,True
3,4x4,MS,7.176949e-14,True
4,6x6,SS,5.836658e-14,True
5,6x6,SS,6.631337e-14,True
6,6x6,MS,6.304288e-13,True
7,6x6,MS,5.038570e-13,True
8,8x8,SS,5.095009e-14,True
9,8x8,SS,2.353065e-14,True


## 안정성 지표 요약

`feasible_fraction`은 전체 read 중 원래 CFLP formulation 기준으로 feasible한 sample의 비율이다.

In [4]:
sa_results[
    [
        "instance",
        "formulation",
        "linking",
        "qubo_variables",
        "feasible_fraction",
        "best_feasible_objective",
        "runtime",
    ]
]

,instance,formulation,linking,qubo_variables,feasible_fraction,best_feasible_objective,runtime
0,4x4,SS,False,44,0.009,2593.0879,0.419581
1,4x4,SS,True,60,0.008,2416.8364,0.568637
2,4x4,MS,False,120,0.897,3123.4155,2.206470
3,4x4,MS,True,212,0.630,3578.5833,3.500570
4,6x6,SS,False,79,0.002,5988.4188,0.734470
5,6x6,SS,True,115,0.003,5711.6042,1.045273
6,6x6,MS,False,259,0.977,5193.7256,6.466007
7,6x6,MS,True,475,0.653,5930.6206,9.536324
8,8x8,SS,False,117,0.002,5271.9611,1.185111
9,8x8,SS,True,181,0.003,5962.6765,1.715565


## linking constraint 비교

세 가지 비교를 수행한다.

1. **linking 유무** — 같은 formulation 안에서 포함/제외를 비교한다. feasible region이 동일하므로 차이는 전부 QUBO 표현의 비용이다.
2. **linking 포함끼리** — SS vs MS
3. **linking 제외끼리** — SS vs MS (본 실험의 기본 설정)

In [5]:
from src.comparison import linking_effect, formulation_gap

print("[1] linking 유무 (같은 formulation 내)")
display(linking_effect(sa_results))

[1] linking 유무 (같은 formulation 내)


,instance,size,formulation,vars_without,vars_with,vars_ratio,terms_without,terms_with,terms_ratio,feasible_without,feasible_with,objective_without,objective_with,runtime_without,runtime_with,status_without,status_with
1,4x4,4,MS,120,212,1.77,2540,3384,1.33,0.897,0.630,3123.4155,3578.5833,2.206470,3.500570,OK,OK
3,6x6,6,MS,259,475,1.83,8701,10753,1.24,0.977,0.653,5193.7256,5930.6206,6.466007,9.536324,OK,OK
5,8x8,8,MS,413,773,1.87,17603,20843,1.18,0.909,0.597,6683.2315,6410.8175,11.762937,17.559279,OK,OK
7,15x15,15,MS,1439,2774,1.93,123857,136427,1.10,0.999,0.588,14477.0340,14505.1572,66.175773,86.225369,OK,OK
0,4x4,4,SS,44,60,1.36,246,278,1.13,0.009,0.008,2593.0879,2416.8364,0.419581,0.568637,OK,OK
2,6x6,6,SS,79,115,1.46,571,643,1.13,0.002,0.003,5988.4188,5711.6042,0.734470,1.045273,OK,OK
4,8x8,8,SS,117,181,1.55,1030,1158,1.12,0.002,0.003,5271.9611,5962.6765,1.185111,1.715565,OK,OK
6,15x15,15,SS,329,554,1.68,5026,5476,1.09,0.000,0.000,NaN,NaN,3.645963,5.430555,OK,OK


In [6]:
print("[2] linking 포함끼리: SS vs MS")
display(formulation_gap(sa_results, linking=True))
print("[3] linking 제외끼리: SS vs MS")
display(formulation_gap(sa_results, linking=False))

[2] linking 포함끼리: SS vs MS


,instance,size,linking,vars_SS,vars_MS,vars_MS_over_SS,terms_SS,terms_MS,terms_MS_over_SS,feasible_SS,feasible_MS,objective_SS,objective_MS,status_SS,status_MS
0,4x4,4,True,60,212,3.53,278,3384,12.17,0.008,0.630,2416.8364,3578.5833,OK,OK
1,6x6,6,True,115,475,4.13,643,10753,16.72,0.003,0.653,5711.6042,5930.6206,OK,OK
2,8x8,8,True,181,773,4.27,1158,20843,18.00,0.003,0.597,5962.6765,6410.8175,OK,OK
3,15x15,15,True,554,2774,5.01,5476,136427,24.91,0.000,0.588,NaN,14505.1572,OK,OK


[3] linking 제외끼리: SS vs MS


,instance,size,linking,vars_SS,vars_MS,vars_MS_over_SS,terms_SS,terms_MS,terms_MS_over_SS,feasible_SS,feasible_MS,objective_SS,objective_MS,status_SS,status_MS
0,4x4,4,False,44,120,2.73,246,2540,10.33,0.009,0.897,2593.0879,3123.4155,OK,OK
1,6x6,6,False,79,259,3.28,571,8701,15.24,0.002,0.977,5988.4188,5193.7256,OK,OK
2,8x8,8,False,117,413,3.53,1030,17603,17.09,0.002,0.909,5271.9611,6683.2315,OK,OK
3,15x15,15,False,329,1439,4.37,5026,123857,24.64,0.000,0.999,NaN,14477.0340,OK,OK
